In [9]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, Bidirectional, LSTM, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
import xgboost as xgb
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import make_pipeline


# Load MIT-BIH Arrhythmia Dataset (Training and Test sets)
mitbih_train = pd.read_csv('mitbih_train.csv', header=None)
mitbih_test = pd.read_csv('mitbih_test.csv', header=None)

# Separate features and labels for MIT-BIH Dataset
X_mitbih_train = mitbih_train.iloc[:, :-1].values  
y_mitbih_train = mitbih_train.iloc[:, -1].values   
X_mitbih_test = mitbih_test.iloc[:, :-1].values    
y_mitbih_test = mitbih_test.iloc[:, -1].values   

In [12]:
# Define the LSTM Model for Feature Extraction
def create_lstm_model():
    inputs = Input(shape=(187, 1))
    x = LSTM(16, return_sequences=True)(inputs)
    x = LSTM(8)(x)
    x = Dense(32, activation='relu')(x)
    model = Model(inputs=inputs, outputs=x)
    return model

In [13]:
# Initialize Stratified K-Fold
skf = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)

fold_no = 1
batch_size = 128  # Reduced batch size to lower memory consumption
final_predictions = []
final_true_labels = []

# Iterate through each fold
for train_index, val_index in skf.split(X_mitbih_train, y_mitbih_train):
    print(f"Training fold {fold_no}...")

    # Split the data into training and validation sets for this fold
    X_train_fold, X_val_fold = X_mitbih_train[train_index], X_mitbih_train[val_index]
    y_train_fold, y_val_fold = y_mitbih_train[train_index], y_mitbih_train[val_index]

    # Apply SMOTE to the training fold only
    smote = SMOTE(random_state=42)
    X_train_fold_smote, y_train_fold_smote = smote.fit_resample(X_train_fold, y_train_fold)

    # Standardize data
    scaler = StandardScaler()
    X_train_fold_smote = scaler.fit_transform(X_train_fold_smote)
    X_val_fold = scaler.transform(X_val_fold)

    # Reshape the data for LSTM
    X_train_fold_reshaped = X_train_fold_smote.reshape(-1, 187, 1)
    X_val_fold_reshaped = X_val_fold.reshape(-1, 187, 1)

    # Create the LSTM model for feature extraction
    lstm_model = create_lstm_model()

    # Train the LSTM model
    lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)  # Early stopping with lower patience
    lstm_model.fit(X_train_fold_reshaped, y_train_fold_smote, epochs=5, batch_size=batch_size, validation_data=(X_val_fold_reshaped, y_val_fold), callbacks=[early_stopping], verbose=1)

    # Extract features from the LSTM model
    feature_extractor = Model(inputs=lstm_model.input, outputs=lstm_model.layers[-1].output)
    X_train_features = feature_extractor.predict(X_train_fold_reshaped)
    X_val_features = feature_extractor.predict(X_val_fold_reshaped)

    # Train XGBoost on extracted features
    xgb_model = xgb.XGBClassifier(objective='multi:softmax', num_class=5, eval_metric='mlogloss', use_label_encoder=False)
    xgb_model.fit(X_train_features, y_train_fold_smote)

    # Make predictions using XGBoost model
    y_val_pred_xgb = xgb_model.predict(X_val_features)

    # Train LDA on the combined LSTM and XGBoost predictions
    lda = LinearDiscriminantAnalysis()
    lda_pipeline = make_pipeline(scaler, lda)
    lda_pipeline.fit(X_train_features, y_train_fold_smote)
    y_val_pred_lda = lda_pipeline.predict(X_val_features)

    # Store the predictions and true labels
    final_predictions.extend(y_val_pred_lda)
    final_true_labels.extend(y_val_fold)

    # Print classification report for the current fold
    print(f"Classification Report for Fold {fold_no}:\n", classification_report(y_val_fold, y_val_pred_lda, target_names=['N', 'S', 'V', 'F', 'Q']))

    # Clear the session to free up memory
    tf.keras.backend.clear_session()

    fold_no += 1

Training fold 1...
Epoch 1/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 81s 164ms/step - accuracy: 0.2364 - loss: 2.2772
Epoch 2/10
115/472 ━━━━━━━━━━━━━━━━━━━━ 1:00 170ms/step - accuracy: 0.4136 - loss: 1.3743

KeyboardInterrupt: 

In [ ]:
# Evaluate the final model on the MIT-BIH Test Set
X_mitbih_test = scaler.transform(X_mitbih_test).reshape(X_mitbih_test.shape[0], 187, 1)
X_test_features = feature_extractor.predict(X_mitbih_test)
y_test_pred_lda = lda_pipeline.predict(X_test_features)

# Print classification report for the test set
print("Final Classification Report on MIT-BIH Test Set:\n", classification_report(y_mitbih_test, y_test_pred_lda, target_names=['N', 'S', 'V', 'F', 'Q']))